#**USCIS Form Collection Pipeline**

This notebook crawls the USCIS Forms website, discovers publicly available form PDFs, downloads the documents, and genereates some metadata that is used by the document classification pipeline.

## Install + Import Required Libraries

In [ ]:
!pip install beautifulsoup4 requests lxml tqdm

In [ ]:
import os
import requests

from bs4 import BeautifulSoup
from urllib.parse import urljoin
from tqdm import tqdm

## Configure + Test Crawler Settings

In [ ]:
USCIS_FORMS_URL = "https://www.uscis.gov/forms/all-forms"

print(USCIS_FORMS_URL)

https://www.uscis.gov/forms/all-forms


In [ ]:
response = requests.get(USCIS_FORMS_URL)

print(response.status_code)

200


In [ ]:
soup = BeautifulSoup(response.text, "lxml")

print(soup.title.text)

All Forms | USCIS


In [ ]:
links = soup.find_all("a", href=True)

print(f"Number of links found: {len(links)}")

Number of links found: 383


In [ ]:
form_links = []

for link in links:
    href = link["href"]

    if "forms" in href.lower():
        form_links.append(href)

print(f"Found {len(form_links)} links containing 'forms'\n")

for link in form_links[:50]:
    print(link)

Found 13 links containing 'forms'

/forms/forms
/forms/all-forms
/forms/explore-my-options
/forms/forms
/forms/all-forms
/forms/explore-my-options
/forms/filing-guidance
/forms/filing-fees
/forms/forms-updates
/forms/department-of-state-ds-and-other-non-uscis-forms
/forms/forms
https://www.uscis.gov/forms/all-forms/g-325r
/forms/forms


## Helper Functions

In [ ]:
import time
import requests

USER_AGENT = (
    "NewStartAI-Capstone-Crawler/0.1 "
    "(Educational Use)"
)

REQUEST_DELAY_SECONDS = 2

def polite_get(url):
    """
    Makes a respectful request to a website.
    """

    time.sleep(REQUEST_DELAY_SECONDS)

    response = requests.get(
        url,
        headers={"User-Agent": USER_AGENT},
        timeout=30
    )

    response.raise_for_status()

    return response

In [ ]:
response = polite_get("https://www.uscis.gov/forms/all-forms")

print(response.status_code)

200


In [ ]:
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
import re

In [ ]:
FORM_DETAIL_PATH_RE = re.compile(
    r"^/[a-z]{1,2}-\d+[a-z0-9]*$",
    re.IGNORECASE
)


def is_form_detail_path(path):
    return bool(FORM_DETAIL_PATH_RE.match(path))

In [ ]:
test_paths = [
    "/i-485",
    "/n-400",
    "/i-765",
    "/forms/all-forms",
    "/forms/filing-guidance",
    "/topics"
]


for path in test_paths:
    print(path, "->", is_form_detail_path(path))

/i-485 -> True
/n-400 -> True
/i-765 -> True
/forms/all-forms -> False
/forms/filing-guidance -> False
/topics -> False


In [ ]:
id="d2r6kd"
def find_form_detail_links(index_url):
    """
    Finds USCIS form-detail pages from the All Forms index page.
    """

    response = polite_get(index_url)

    soup = BeautifulSoup(response.text, "lxml")

    form_links = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        # Convert relative URLs into full URLs
        full_url = urljoin("https://www.uscis.gov", href)

        # Extract only the path portion
        path = urlparse(full_url).path

        # Keep only form pages
        if is_form_detail_path(path):
            form_links.append(full_url)

    # Remove duplicates while preserving order
    form_links = list(dict.fromkeys(form_links))

    return form_links

## Find USCIS Form Pages

The crawler begins at the USCIS "All Forms" page and identifies individual form pages that match the expected URL pattern.

In [ ]:
id="y2x3mf"
USCIS_FORMS_URL = "https://www.uscis.gov/forms/all-forms"

detail_links = find_form_detail_links(USCIS_FORMS_URL)

print(f"Found {len(detail_links)} form pages")

print("\nFirst 10:")
for link in detail_links[:10]:
    print(link)

Found 102 form pages

First 10:
https://www.uscis.gov/i-9
https://www.uscis.gov/i-485
https://www.uscis.gov/i-765
https://www.uscis.gov/i-90
https://www.uscis.gov/n-400
https://www.uscis.gov/i-129f
https://www.uscis.gov/i-130
https://www.uscis.gov/i-360
https://www.uscis.gov/i-600
https://www.uscis.gov/i-751


In [ ]:
def find_pdf_links(detail_url):
    """
    Finds PDF links on a USCIS form detail page.
    """

    response = polite_get(detail_url)

    soup = BeautifulSoup(response.text, "lxml")

    pdf_links = []

    for a in soup.find_all("a", href=True):

        href = a["href"]

        if href.lower().split("?")[0].endswith(".pdf"):

            full_url = urljoin(detail_url, href)

            pdf_links.append(full_url)

    # Remove duplicates
    pdf_links = list(dict.fromkeys(pdf_links))

    return pdf_links

In [ ]:
sample_form = "https://www.uscis.gov/i-485"

pdfs = find_pdf_links(sample_form)

print(f"Found {len(pdfs)} PDFs\n")

for pdf in pdfs:
    print(pdf)

Found 2 PDFs

https://www.uscis.gov/sites/default/files/document/forms/i-485.pdf
https://www.uscis.gov/sites/default/files/document/forms/i-485instr.pdf


## Collect PDF Links
Each form page is scanned for downloadable PDF documents including forms, instructions, supplements, and translated versions.

In [ ]:
def discover_all_pdf_links(form_pages):
    """
    Finds all PDF documents from all USCIS form pages.
    """

    all_pdfs = []

    for i, page in enumerate(form_pages):

        print(f"Processing {i+1}/{len(form_pages)}: {page}")

        pdfs = find_pdf_links(page)

        all_pdfs.extend(pdfs)

    # remove duplicates
    all_pdfs = list(dict.fromkeys(all_pdfs))

    return all_pdfs

In [ ]:
uscis_pdf_links = discover_all_pdf_links(detail_links)

print("\nTotal PDFs found:", len(uscis_pdf_links))

print("\nFirst 20 PDFs:")
for pdf in uscis_pdf_links[:20]:
    print(pdf)

Processing 1/102: https://www.uscis.gov/i-9
Processing 2/102: https://www.uscis.gov/i-485
Processing 3/102: https://www.uscis.gov/i-765
Processing 4/102: https://www.uscis.gov/i-90
Processing 5/102: https://www.uscis.gov/n-400
Processing 6/102: https://www.uscis.gov/i-129f
Processing 7/102: https://www.uscis.gov/i-130
Processing 8/102: https://www.uscis.gov/i-360
Processing 9/102: https://www.uscis.gov/i-600
Processing 10/102: https://www.uscis.gov/i-751
Processing 11/102: https://www.uscis.gov/i-129
Processing 12/102: https://www.uscis.gov/i-140
Processing 13/102: https://www.uscis.gov/i-526
Processing 14/102: https://www.uscis.gov/i-539
Processing 15/102: https://www.uscis.gov/i-589
Processing 16/102: https://www.uscis.gov/i-730
Processing 17/102: https://www.uscis.gov/i-821
Processing 18/102: https://www.uscis.gov/ar-11
Processing 19/102: https://www.uscis.gov/g-28
Processing 20/102: https://www.uscis.gov/g-28i
Processing 21/102: https://www.uscis.gov/g-325a
Processing 22/102: https

## Download USCIS PDFs
Download all discovered PDF documents

In [ ]:
import os
from pathlib import Path


DATA_DIR = Path("uscis_pdfs")

DATA_DIR.mkdir(exist_ok=True)


def download_pdf(url, save_dir=DATA_DIR):

    filename = url.split("/")[-1]

    filepath = save_dir / filename

    if filepath.exists():
        return filepath

    response = polite_get(url)

    with open(filepath, "wb") as f:
        f.write(response.content)

    return filepath

In [ ]:
sample_downloads = []

for url in uscis_pdf_links[:5]:

    path = download_pdf(url)
    sample_downloads.append(path)

    print("Downloaded:", path)

Downloaded: uscis_pdfs/i-9.pdf
Downloaded: uscis_pdfs/i-9instr.pdf
Downloaded: uscis_pdfs/i-9-spanish.pdf
Downloaded: uscis_pdfs/i9-INS-Spanish.pdf
Downloaded: uscis_pdfs/i-485.pdf


In [ ]:
from pathlib import Path

for file in Path("uscis_pdfs").iterdir():
    print(file.name, "----", file.stat().st_size/1024, "KB")

i-9instr.pdf ---- 290.0556640625 KB
i-9-spanish.pdf ---- 592.7978515625 KB
i9-INS-Spanish.pdf ---- 404.5537109375 KB
i-485.pdf ---- 1175.435546875 KB
i-9.pdf ---- 511.8115234375 KB


In [ ]:
!pip install pymupdf

## Extract text
Opens pdf files and extracts the actual text from each file

In [ ]:
import fitz  # PyMuPDF


def extract_text(pdf_path):

    doc = fitz.open(pdf_path)

    text = ""

    for page in doc:
        text += page.get_text()

    return text


text = extract_text("uscis_pdfs/i-485.pdf")

print(text[:1000])

Form I-485   Edition   01/20/25 
  Page 1 of 24
Section of Law
Application to Register Permanent Residence 
or Adjust Status 
Department of Homeland Security 
U.S. Citizenship and Immigration Services 
USCIS 
Form I-485 
 OMB No. 1615-0023
Expires 10/31/2027
Priority Date:
Country Chargeable:
INA 209(a)
INA 249
INA 245(a)
INA 245(i)
INA 245(m)
Sec. 13, Act of 9/11/57
Cuban Adjustment Act
Other 
Lawful Permanent 
Resident as of:
Date of  
Initial Interview:
Action Block
Receipt
Interview 
Waived
Applicant 
Interviewed
Date Form I-693 Signed By Civil Surgeon:
Attorney State Bar Number 
(if applicable)
Select this box if 
Form G-28 is 
attached.
Volag Number    
(if any)                    
Attorney or Accredited Representative 
USCIS Online Account Number (if any)
To be completed by an Attorney or Accredited Representative (if any).
For USCIS Use Only 
START HERE - Type or print in black ink.
►
NOTE TO ALL APPLICANTS: If you do not completely fill out this application or fail to submit r

In [ ]:
import fitz
from pathlib import Path


def extract_text(pdf_path):
    """
    Extract text from a PDF using PyMuPDF.
    """

    try:
        doc = fitz.open(pdf_path)

        text = ""

        for page in doc:
            text += page.get_text()

        doc.close()

        return text.strip()

    except Exception as e:
        print(f"Error extracting {pdf_path}: {e}")
        return ""

In [ ]:
for pdf in Path("uscis_pdfs").iterdir():

    text = extract_text(pdf)

    print("\n----------------")
    print(pdf.name)
    print("Characters:", len(text))
    print(text[:200])


----------------
i-9instr.pdf
Characters: 29101
Page 1 of 8
Form I-9 Instructions   01/20/25
Instructions for Form I-9, 
Employment Eligibility Verification                      
Department of Homeland Security 
U.S. Citizenship and Immigration Ser

----------------
i-9-spanish.pdf
Characters: 16705
USCIS 
Formulario I-9
OMB No. 1615-0047 
Expire 05/31/2027 
Verificación de Elegibilidad de Empleo 
Departamento de Seguridad Nacional 
Servicio de Ciudadanía e Inmigracion de Estados Unidos 
COMIENCE

----------------
i9-INS-Spanish.pdf
Characters: 32247
Page 1 of 8
Form I-9 Instructions   01/20/25
Instrucciones para el Formulario I-9,  
Verificación de Elegibilidad de Empleo                                                    
Departamento de Segurida

----------------
i-485.pdf
Characters: 54553
Form I-485   Edition   01/20/25 
  Page 1 of 24
Section of Law
Application to Register Permanent Residence 
or Adjust Status 
Department of Homeland Security 
U.S. Citizenship and Immigration Ser

In [ ]:
from pathlib import Path

for pdf in sorted(Path("uscis_pdfs").iterdir()):
    print(pdf.name)

i-485.pdf
i-9-spanish.pdf
i-9.pdf
i-9instr.pdf
i9-INS-Spanish.pdf


In [ ]:
from pathlib import Path

FULL_DATA_DIR = Path("uscis_all_pdfs")
FULL_DATA_DIR.mkdir(exist_ok=True)


def download_pdf(url, save_dir=FULL_DATA_DIR):

    filename = url.split("/")[-1]

    filepath = save_dir / filename

    if filepath.exists():
        return filepath

    response = polite_get(url)

    with open(filepath, "wb") as f:
        f.write(response.content)

    return filepath


In [ ]:
from urllib.parse import urlparse

def is_valid_uscis_form_pdf(url):
    parsed = urlparse(url)

    return (
        parsed.netloc.endswith("uscis.gov")
        and "/document/forms/" in parsed.path
        and parsed.path.lower().endswith(".pdf")
    )


filtered_uscis_pdf_links = [
    url for url in uscis_pdf_links
    if is_valid_uscis_form_pdf(url)
]

print("Before filtering:", len(uscis_pdf_links))
print("After filtering:", len(filtered_uscis_pdf_links))

filtered_uscis_pdf_links[:10]

Before filtering: 281
After filtering: 256


['https://www.uscis.gov/sites/default/files/document/forms/i-9.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-9-spanish.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i9-INS-Spanish.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-485.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-485instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765instr.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-765ws.pdf',
 'https://www.uscis.gov/sites/default/files/document/forms/i-90.pdf']

## Build Metadata
Metadata is used to describe each document from the filenames and is saved for processing down the line

In [ ]:
import pandas as pd
import re
from urllib.parse import urlparse


def extract_form_number(filename):
    """
    Extract USCIS form identifier.
    Examples:
    i-485.pdf -> I-485
    n-400instr.pdf -> N-400
    g-28.pdf -> G-28
    """

    match = re.match(r"([a-z]+-\d+)", filename.lower())

    if match:
        return match.group(1).upper()

    return None


def classify_document_type(filename):

    name = filename.lower()

    if "instr" in name:
        return "instructions"

    if "sup" in name:
        return "supplement"

    if "checklist" in name:
        return "checklist"

    if "spanish" in name or "es" in name:
        return "translated_form"

    return "form"


uscis_metadata = []

for url in filtered_uscis_pdf_links:

    filename = url.split("/")[-1]

    uscis_metadata.append({
        "filename": filename,
        "agency": "USCIS",
        "form_number": extract_form_number(filename),
        "document_type": classify_document_type(filename),
        "url": url
    })


uscis_df = pd.DataFrame(uscis_metadata)

uscis_df.head(10)

,filename,agency,form_number,document_type,url
0,i-9.pdf,USCIS,I-9,form,https://www.uscis.gov/sites/default/files/docu...
1,i-9instr.pdf,USCIS,I-9,instructions,https://www.uscis.gov/sites/default/files/docu...
2,i-9-spanish.pdf,USCIS,I-9,translated_form,https://www.uscis.gov/sites/default/files/docu...
3,i9-INS-Spanish.pdf,USCIS,None,translated_form,https://www.uscis.gov/sites/default/files/docu...
4,i-485.pdf,USCIS,I-485,form,https://www.uscis.gov/sites/default/files/docu...
5,i-485instr.pdf,USCIS,I-485,instructions,https://www.uscis.gov/sites/default/files/docu...
6,i-765.pdf,USCIS,I-765,form,https://www.uscis.gov/sites/default/files/docu...
7,i-765instr.pdf,USCIS,I-765,instructions,https://www.uscis.gov/sites/default/files/docu...
8,i-765ws.pdf,USCIS,I-765,form,https://www.uscis.gov/sites/default/files/docu...
9,i-90.pdf,USCIS,I-90,form,https://www.uscis.gov/sites/default/files/docu...


In [ ]:
uscis_df["document_type"].value_counts()

,count
document_type,
form,123
instructions,110
supplement,14
checklist,6
translated_form,3


In [ ]:
uscis_df["form_number"].value_counts().head(20)

,count
form_number,
I-589,26
I-129,12
I-601,10
I-956,10
I-800,8
I-600,7
I-864,6
I-485,6
I-914,5


In [ ]:
uscis_df[uscis_df["form_number"].isin(
    ["I-589", "I-129", "I-601", "I-956"]
)][["filename","form_number","document_type"]]

,filename,form_number,document_type
13,i-129f.pdf,I-129,form
14,i-129finstr.pdf,I-129,instructions
28,i-129.pdf,I-129,form
29,i-129instr.pdf,I-129,instructions
30,i-129h2a.pdf,I-129,form
31,i-129h2ainstr.pdf,I-129,instructions
44,i-589.pdf,I-589,form
45,i-589instr.pdf,I-589,instructions
46,I-589%20Form%20with%20Watermark_AR.pdf,I-589,form
47,I-589%20Instruction_AR.pdf,I-589,instructions


In [ ]:
uscis_df.to_csv("uscis_metadata.csv", index=False)

print("Saved:", len(uscis_df), "rows")

Saved: 256 rows


## Export Dataset

In [ ]:
import shutil
from pathlib import Path

source = Path("/content/uscis_pdfs")

destination = Path("/content/drive/MyDrive/newstart_ai/data/raw/uscis")

destination.mkdir(parents=True, exist_ok=True)


for file in source.glob("*.pdf"):
    shutil.copy(file, destination / file.name)

print("Copied USCIS PDFs:", len(list(destination.glob("*.pdf"))))

Copied USCIS PDFs: 5


In [ ]:
from pathlib import Path

uscis_drive = Path("/content/drive/MyDrive/newstart_ai/data/raw/uscis")

for file in uscis_drive.glob("*.pdf"):
    file.unlink()

print("Remaining USCIS files:", len(list(uscis_drive.glob("*.pdf"))))

Remaining USCIS files: 0


In [ ]:
from pathlib import Path

uscis_download_dir = Path("/content/uscis_pdfs")

# Clear old test downloads
for file in uscis_download_dir.glob("*.pdf"):
    file.unlink()

downloaded_files = []

for i, url in enumerate(filtered_uscis_pdf_links):

    try:
        path = download_pdf(url)
        downloaded_files.append(path)
        print(f"{i+1}/{len(filtered_uscis_pdf_links)} downloaded: {Path(path).name}")

    except Exception as e:
        print(f"FAILED: {url}")
        print(e)

print("\nDownload complete")
print("Downloaded:", len(downloaded_files))

1/256 downloaded: i-9.pdf
2/256 downloaded: i-9instr.pdf
3/256 downloaded: i-9-spanish.pdf
4/256 downloaded: i9-INS-Spanish.pdf
5/256 downloaded: i-485.pdf
6/256 downloaded: i-485instr.pdf
7/256 downloaded: i-765.pdf
8/256 downloaded: i-765instr.pdf
9/256 downloaded: i-765ws.pdf
10/256 downloaded: i-90.pdf
11/256 downloaded: i-90instr.pdf
12/256 downloaded: n-400.pdf
13/256 downloaded: n-400instr.pdf
14/256 downloaded: i-129f.pdf
15/256 downloaded: i-129finstr.pdf
16/256 downloaded: i-130.pdf
17/256 downloaded: i-130instr.pdf
18/256 downloaded: i-130a.pdf
19/256 downloaded: i-360.pdf
20/256 downloaded: i-360instr.pdf
21/256 downloaded: m-737.pdf
22/256 downloaded: i-600.pdf
23/256 downloaded: i-600instr.pdf
24/256 downloaded: i-600asup1.pdf
25/256 downloaded: i-600asup2.pdf
26/256 downloaded: i-600asup3.pdf
27/256 downloaded: i-751.pdf
28/256 downloaded: i-751instr.pdf
29/256 downloaded: i-129.pdf
30/256 downloaded: i-129instr.pdf
31/256 downloaded: i-129h2a.pdf
32/256 downloaded: i-12

In [ ]:
import shutil
from pathlib import Path

source = Path("/content/uscis_all_pdfs")

destination = Path("/content/drive/MyDrive/newstart_ai/data/raw/uscis")

destination.mkdir(parents=True, exist_ok=True)

for file in source.glob("*.pdf"):
    shutil.copy(file, destination / file.name)

print(
    "Copied USCIS PDFs:",
    len(list(destination.glob("*.pdf")))
)

Copied USCIS PDFs: 256


In [ ]:
uscis_df.to_csv(
    "/content/drive/MyDrive/newstart_ai/data/raw/uscis_metadata.csv",
    index=False
)

print("Saved USCIS metadata")

Saved USCIS metadata


In [ ]:
import pandas as pd
from pathlib import Path
import re


# Folder where your USCIS PDFs are stored
uscis_folder = Path("/content/uscis_all_pdfs")


def extract_form_number(filename):
    """
    Extract USCIS form number.
    Examples:
    i-485.pdf -> I-485
    n-400.pdf -> N-400
    g-28.pdf -> G-28
    """

    match = re.match(r"([a-z]+-\d+)", filename.lower())

    if match:
        return match.group(1).upper()

    return None


def classify_document_type(filename):

    name = filename.lower()

    if "instr" in name:
        return "instructions"

    if "sup" in name:
        return "supplement"

    if "checklist" in name:
        return "checklist"

    if "spanish" in name or "es" in name:
        return "translated_form"

    return "form"


uscis_metadata = []


for pdf in uscis_folder.glob("*.pdf"):

    filename = pdf.name

    uscis_metadata.append({
        "filename": filename,
        "agency": "USCIS",
        "form_number": extract_form_number(filename),
        "document_type": classify_document_type(filename),
        "filepath": str(pdf)
    })


uscis_df = pd.DataFrame(uscis_metadata)


# Save CSV
uscis_df.to_csv(
    "uscis_metadata.csv",
    index=False
)


print("USCIS metadata saved!")
print("Rows:", len(uscis_df))

uscis_df.head()

USCIS metadata saved!
Rows: 256


,filename,agency,form_number,document_type,filepath
0,i-765ws.pdf,USCIS,I-765,form,/content/uscis_all_pdfs/i-765ws.pdf
1,i-361instr.pdf,USCIS,I-361,instructions,/content/uscis_all_pdfs/i-361instr.pdf
2,i-508instr.pdf,USCIS,I-508,instructions,/content/uscis_all_pdfs/i-508instr.pdf
3,g-1566instr.pdf,USCIS,G-1566,instructions,/content/uscis_all_pdfs/g-1566instr.pdf
4,i-129s.pdf,USCIS,I-129,form,/content/uscis_all_pdfs/i-129s.pdf


In [ ]:
from pathlib import Path

raw = Path("/content/drive/MyDrive/newstart_ai/data/raw")

for folder in ["uscis", "dmv", "ssa", "irs"]:
    path = raw / folder

    print("\n", folder)
    print("Exists:", path.exists())

    if path.exists():
        print("PDF count:", len(list(path.glob("*.pdf"))))


 uscis
Exists: True
PDF count: 0

 dmv
Exists: True
PDF count: 277

 ssa
Exists: False

 irs
Exists: True
PDF count: 0


In [ ]:
import shutil
from pathlib import Path

source = Path("/content/uscis_all_pdfs")

destination = Path("/content/drive/MyDrive/newstart_ai/data/raw/uscis")

destination.mkdir(parents=True, exist_ok=True)

count = 0

for file in source.glob("*.pdf"):
    shutil.copy(file, destination / file.name)
    count += 1

print("Copied USCIS PDFs:", count)

Copied USCIS PDFs: 256


In [ ]:
import pandas as pd
import re
from pathlib import Path


uscis_folder = Path("/content/drive/MyDrive/newstart_ai/data/raw/uscis")


def extract_form_number(filename):
    """
    Extract USCIS form identifier.
    Examples:
    i-485.pdf -> I-485
    n-400instr.pdf -> N-400
    g-28.pdf -> G-28
    """

    match = re.match(r"([a-z]+-\d+)", filename.lower())

    if match:
        return match.group(1).upper()

    return None


def classify_document_type(filename):

    name = filename.lower()

    if "instr" in name:
        return "instructions"

    if "sup" in name:
        return "supplement"

    if "checklist" in name:
        return "checklist"

    if "spanish" in name or "es" in name:
        return "translated_form"

    return "form"


metadata = []

for pdf in uscis_folder.glob("*.pdf"):

    metadata.append({
        "filename": pdf.name,
        "filepath": str(pdf),
        "agency": "USCIS",
        "form_number": extract_form_number(pdf.name),
        "document_type": classify_document_type(pdf.name)
    })


uscis_df = pd.DataFrame(metadata)

uscis_df.head()

,filename,filepath,agency,form_number,document_type
0,i-765ws.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-765,form
1,i-361instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-361,instructions
2,i-508instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-508,instructions
3,g-1566instr.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,G-1566,instructions
4,i-129s.pdf,/content/drive/MyDrive/newstart_ai/data/raw/us...,USCIS,I-129,form


In [ ]:
metadata_folder = Path("/content/drive/MyDrive/newstart_ai/data/metadata")

metadata_folder.mkdir(parents=True, exist_ok=True)

In [ ]:
uscis_csv = metadata_folder / "uscis_metadata.csv"

uscis_df.to_csv(uscis_csv, index=False)

print("Saved:", uscis_csv)
print("Rows:", len(uscis_df))

Saved: /content/drive/MyDrive/newstart_ai/data/metadata/uscis_metadata.csv
Rows: 256
